In [22]:
import pandas as pd
import numpy as np
import os

In [23]:


class NeuronOverlapAnalyzer:
    """神经元重叠分析器"""
    
    def __init__(self, root_path, results_folder="Results"):
        self.root_path = root_path
        self.results_path = os.path.join(root_path, results_folder)
        os.makedirs(self.results_path, exist_ok=True)
    
    def load_onoff_data(self, mouse_name, session_numbers):
        """加载ON/OFF数据"""
        data_dict = {}
        
        for session_num in session_numbers:
            file_path = os.path.join(self.results_path, mouse_name, f"session_{session_num}_ONOFF.csv")
            if os.path.exists(file_path):
                try:
                    df = pd.read_csv(file_path)
                    data_dict[f"session_{session_num}"] = df
                    print(f"成功加载: session_{session_num}_ONOFF.csv (形状: {df.shape})")
                except Exception as e:
                    print(f"加载失败 {file_path}: {e}")
            else:
                print(f"文件不存在: {file_path}")
        
        return data_dict
    
    def extract_on_neurons_from_df(self, df, session_name=""):
        """从DataFrame中提取ON神经元"""
        # 检查DataFrame结构
        print(f"\n分析 {session_name}:")
        print(f"  列名: {df.columns.tolist()}")
        print(f"  前几行索引: {df.index.tolist()[:5]}")
        
        # 识别神经元编号列
        neuron_col = None
        if 'Unnamed: 0' in df.columns:
            neuron_col = 'Unnamed: 0'
        elif 'Neurons' in df.columns:
            neuron_col = 'Neurons'
        elif df.columns[0] == 'Frame' or df.iloc[0, 0] == 'Frame':
            neuron_col = df.columns[0]
        
        if neuron_col is None:
            print(f"  警告: 未找到神经元编号列")
            return []
        
        print(f"  神经元编号列: {neuron_col}")
        
        # 找到所有以"_ON"结尾的列
        on_columns = [col for col in df.columns if col.endswith('_ON')]
        print(f"  ON列: {on_columns}")
        
        if not on_columns:
            print(f"  警告: 没有找到以'_ON'结尾的列")
            return []
        
        # 提取ON神经元
        on_neurons = set()
        
        for col in on_columns:
            # 找到该ON列为1的行
            mask = df[col] == 1
            
            # 获取对应的神经元编号
            neurons_in_col = df.loc[mask, neuron_col].tolist()
            
            # 过滤C开头的神经元
            c_neurons = [n for n in neurons_in_col if isinstance(n, str) and n.startswith('C')]
            
            if c_neurons:
                print(f"  列 {col}: {len(c_neurons)} 个神经元")
                on_neurons.update(c_neurons)
        
        result = sorted(on_neurons)
        print(f"  总共找到 {len(result)} 个ON神经元: {result}")
        
        return result
    
    def analyze_overlap(self, mouse_name, session_pairs):
        """分析多个session对的重叠"""
        all_results = {}
        
        for session1, session2 in session_pairs:
            print(f"\n{'='*60}")
            print(f"分析 {mouse_name}: session_{session1} 与 session_{session2}")
            print('='*60)
            
            # 加载数据
            data_dict = self.load_onoff_data(mouse_name, [session1, session2])
            
            if len(data_dict) < 2:
                print("数据不足，跳过此对")
                continue
            
            # 提取神经元
            neurons1 = self.extract_on_neurons_from_df(
                data_dict[f"session_{session1}"], 
                f"session_{session1}"
            )
            neurons2 = self.extract_on_neurons_from_df(
                data_dict[f"session_{session2}"], 
                f"session_{session2}"
            )
            
            # 计算交集和比例
            set1, set2 = set(neurons1), set(neurons2)
            intersection = sorted(set1.intersection(set2))
            
            # 计算比例
            ratio1 = len(intersection) / len(neurons1) * 100 if neurons1 else 0
            ratio2 = len(intersection) / len(neurons2) * 100 if neurons2 else 0
            
            # 保存结果
            result = {
                'session1': session1,
                'session2': session2,
                'neurons1': neurons1,
                'neurons2': neurons2,
                'intersection': intersection,
                'count1': len(neurons1),
                'count2': len(neurons2),
                'intersection_count': len(intersection),
                'ratio1': ratio1,
                'ratio2': ratio2,
                'unique1': sorted(set1 - set2),
                'unique2': sorted(set2 - set1)
            }
            
            all_results[(session1, session2)] = result
            
            # 打印结果
            self.print_overlap_result(result)
            
            # 保存到Excel
            self.save_overlap_result(result, mouse_name, session1, session2)
        
        return all_results
    
    def print_overlap_result(self, result):
        """打印重叠结果"""
        print(f"\n{'='*60}")
        print("重叠分析结果:")
        print('='*60)
        print(f"Session {result['session1']}: {result['count1']} 个ON神经元")
        print(f"Session {result['session2']}: {result['count2']} 个ON神经元")
        print(f"共同ON神经元: {result['intersection_count']} 个")
        print(f"交集: {result['intersection']}")
        print(f"重叠比例: Session {result['session1']} -> {result['ratio1']:.2f}%")
        print(f"重叠比例: Session {result['session2']} -> {result['ratio2']:.2f}%")
        print(f"Session {result['session1']} 独有: {result['unique1']}")
        print(f"Session {result['session2']} 独有: {result['unique2']}")
        
        # 韦恩图式展示
        print(f"\n韦恩图统计:")
        print(f"Session {result['session1']} | Session {result['session2']}")
        print(f"  {len(result['unique1'])}   |   {len(result['unique2'])}")
        print(f"     \\ {result['intersection_count']} /")
        print(f"      共同")
    
    def save_overlap_result(self, result, mouse_name, session1, session2):
        """保存结果到Excel"""
        # 创建对比表格
        all_neurons = sorted(set(result['neurons1'] + result['neurons2']))
        
        comparison_data = []
        for neuron in all_neurons:
            comparison_data.append({
                '神经元': neuron,
                f'session_{session1}': 1 if neuron in result['neurons1'] else 0,
                f'session_{session2}': 1 if neuron in result['neurons2'] else 0
            })
        
        comparison_df = pd.DataFrame(comparison_data)
        
        # 保存到Excel
        save_filename = f"{mouse_name}_session{session1}_vs_session{session2}_overlap.xlsx"
        save_path = os.path.join(self.results_path, save_filename)
        
        with pd.ExcelWriter(save_path, engine='openpyxl') as writer:
            # 保存对比表格
            comparison_df.to_excel(writer, sheet_name='对比表格', index=False)
            
            # 保存汇总信息
            summary_data = {
                '指标': [
                    f'Session {session1} ON神经元数量',
                    f'Session {session2} ON神经元数量',
                    '共同ON神经元数量',
                    f'重叠比例 (Session {session1})',
                    f'重叠比例 (Session {session2})',
                    f'Session {session1} 独有神经元',
                    f'Session {session2} 独有神经元',
                    '共同神经元列表'
                ],
                '值': [
                    result['count1'],
                    result['count2'],
                    result['intersection_count'],
                    f"{result['ratio1']:.2f}%",
                    f"{result['ratio2']:.2f}%",
                    ', '.join(result['unique1']),
                    ', '.join(result['unique2']),
                    ', '.join(result['intersection'])
                ]
            }
            
            summary_df = pd.DataFrame(summary_data)
            summary_df.to_excel(writer, sheet_name='汇总', index=False)
        
        print(f"\n结果已保存到: {save_path}")



In [24]:

# 主程序
if __name__ == "__main__":
    # 配置参数
    root_path = "F:/AAA-ZSY(5)"
    mouse_name = "SYN17-RTIT"
    
    # 要分析的session对
    session_pairs = [
        (2, 5),  # session_2 vs session_5
        # 可以添加更多session对
        # (1, 2),
        # (1, 5),
    ]
    
    # 创建分析器
    analyzer = NeuronOverlapAnalyzer(root_path)
    
    # 执行分析
    results = analyzer.analyze_overlap(mouse_name, session_pairs)
    
    # 如果有多个结果，创建总览表格
    if len(results) > 1:
        overview_data = []
        for (s1, s2), result in results.items():
            overview_data.append({
                'Session对': f'{s1} vs {s2}',
                'Session1数量': result['count1'],
                'Session2数量': result['count2'],
                '共同数量': result['intersection_count'],
                f'Session{s1}比例': f"{result['ratio1']:.2f}%",
                f'Session{s2}比例': f"{result['ratio2']:.2f}%"
            })
        
        overview_df = pd.DataFrame(overview_data)
        overview_save_path = os.path.join(analyzer.results_path, f"{mouse_name}_overlap_overview.xlsx")
        overview_df.to_excel(overview_save_path, index=False)
        print(f"\n总览表格已保存到: {overview_save_path}")


分析 SYN17-RTIT: session_2 与 session_5
成功加载: session_2_ONOFF.csv (形状: (148, 7))
成功加载: session_5_ONOFF.csv (形状: (148, 7))

分析 session_2:
  列名: ['Unnamed: 0', 'free_allogroom_ON', 'free_allogroom_OFF', 'sniff_ON', 'sniff_OFF', 'rear_selfgroom_ON', 'rear_selfgroom_OFF']
  前几行索引: [0, 1, 2, 3, 4]
  神经元编号列: Unnamed: 0
  ON列: ['free_allogroom_ON', 'sniff_ON', 'rear_selfgroom_ON']
  列 free_allogroom_ON: 43 个神经元
  总共找到 43 个ON神经元: ['C000', 'C003', 'C008', 'C010', 'C011', 'C013', 'C014', 'C016', 'C021', 'C022', 'C031', 'C033', 'C035', 'C038', 'C039', 'C040', 'C054', 'C055', 'C057', 'C063', 'C068', 'C072', 'C084', 'C086', 'C093', 'C094', 'C097', 'C099', 'C113', 'C115', 'C119', 'C128', 'C131', 'C133', 'C146', 'C147', 'C151', 'C155', 'C156', 'C162', 'C172', 'C178', 'C180']

分析 session_5:
  列名: ['Unnamed: 0', 'free_allogroom_ON', 'free_allogroom_OFF', 'sniff_ON', 'sniff_OFF', 'rear_selfgroom_ON', 'rear_selfgroom_OFF']
  前几行索引: [0, 1, 2, 3, 4]
  神经元编号列: Unnamed: 0
  ON列: ['free_allogroom_ON', 'sniff_ON